# BiLSTM Emotion Classification Training

**AI Learning Assistant** — This notebook trains a Bidirectional LSTM model on preprocessed student emotion text.

**Inputs:** `dataset/train.csv`, `dataset/validation.csv`, `dataset/test.csv`

**Outputs:**
- `models/bilstm_emotion.keras`
- `models/bilstm_tokenizer.joblib`
- `models/bilstm_training_history.json`
- `assets/bilstm_training_curves.png`
- `assets/bilstm_confusion_matrix.png`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.bilstm_model import *  # noqa: F403
from utils.constants import *  # noqa: F403

## 2. Load Datasets

In [ ]:
train_df, val_df, test_df = load_split_datasets()
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head()

## 3. Handle Empty `processed_text`

Empty strings occur when stopword removal eliminates every token. We **replace** them with the `<EMPTY>` placeholder rather than dropping rows so that:

1. Label counts stay aligned with the saved splits
2. The Tokenizer assigns a fixed index for empty inputs
3. The model can learn an explicit signal for edge cases

In [ ]:
train_df, n_train = handle_empty_processed_text(train_df)
val_df, n_val = handle_empty_processed_text(val_df)
test_df, n_test = handle_empty_processed_text(test_df)
print(f"Replaced empty texts -> Train: {n_train}, Val: {n_val}, Test: {n_test}")

## 4. Tokenization & Sequence Padding

The Keras `Tokenizer` is **fit on training data only** to avoid data leakage. Texts are converted to integer sequences and padded to a uniform length.

In [ ]:
tokenizer = create_and_fit_tokenizer(train_df["processed_text"])
vocab_size = len(tokenizer.word_index) + 1

train_sequences = texts_to_sequences(train_df["processed_text"], tokenizer)
max_len = max(infer_max_sequence_length(train_sequences), MAX_SEQUENCE_LENGTH)

x_train = prepare_features(train_df["processed_text"], tokenizer, max_len)
x_val = prepare_features(val_df["processed_text"], tokenizer, max_len)
x_test = prepare_features(test_df["processed_text"], tokenizer, max_len)

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

print(f"Vocab size: {vocab_size:,} | Max length: {max_len}")
print(f"X_train shape: {x_train.shape}")

## 5. Build BiLSTM Architecture

| Layer | Description |
|-------|-------------|
| Embedding | Maps token indices to dense vectors |
| Bidirectional LSTM | Captures context from both directions |
| Dropout | Regularization |
| Dense (ReLU) | Non-linear feature combination |
| Dense (Softmax) | 5-class emotion output |

In [ ]:
model = build_bilstm_model(vocab_size=vocab_size, max_len=max_len)
model.summary()

## 6. Train with EarlyStopping & ModelCheckpoint

- **Optimizer:** Adam
- **Loss:** SparseCategoricalCrossentropy
- **Metric:** Accuracy

In [ ]:
callbacks = get_training_callbacks(model_path=BILSTM_MODEL_PATH)
history = train_bilstm_model(
    model, x_train, y_train, x_val, y_val,
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks,
)

## 7. Evaluate on Test Set

In [ ]:
label_encoder = load_label_encoder()
eval_results = evaluate_bilstm_model(model, x_test, y_test, label_encoder)
print_training_report(history, eval_results, label_encoder)

## 8. Plot Training Curves & Confusion Matrix

In [ ]:
plot_training_history(history, BILSTM_TRAINING_CURVES_PATH, show=True)
plot_confusion_matrix(
    eval_results["confusion_matrix"],
    list(label_encoder.classes_),
    BILSTM_CONFUSION_MATRIX_PATH,
    show=True,
)

## 9. Save Artifacts

In [ ]:
save_tokenizer(tokenizer, BILSTM_TOKENIZER_PATH)
save_max_len(max_len, BILSTM_MAX_LEN_PATH)
save_training_history(history, BILSTM_HISTORY_PATH)

print(f"Model:     {BILSTM_MODEL_PATH}")
print(f"Tokenizer: {BILSTM_TOKENIZER_PATH}")
print(f"History:   {BILSTM_HISTORY_PATH}")

## 10. Inference Helpers (for later integration)

In [ ]:
sample_texts = train_df["processed_text"].head(3).tolist()
pred_labels, probs = predict_emotion_labels(sample_texts)
pred_names = decode_emotion_labels(pred_labels, label_encoder)

for text, name, prob in zip(sample_texts, pred_names, probs):
    print(f"Text: {text}")
    print(f"  Predicted: {name} (confidence: {prob.max():.2f})\n")